In [1]:

from datasets import load_dataset


#loading from hugging face
dataset = load_dataset("tattabio/ec_classification")
## making test and train set
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
##sanity check to make sure dims are the same as on the DGEB paper dimensions
print(train_df.shape)
print(test_df.shape)

/Users/nehalgarg/690U proj/dgeb_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(512, 3)
(128, 3)


In [2]:
train_df.head()
## checking what the label and seq loomks like 

,Entry,Label,Sequence
0,Q9LQC0,1.14.14.18,MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
1,A0AFT8,1.14.14.18,MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFEL...
2,P74133,1.14.14.18,MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLY...
3,O31534,1.14.14.18,MFVQLRKMTVKEGFADKVIERFSAEGIIEKQEGLIDVTVLEKNVRR...
4,P34697,1.15.1.1,MFMNLLTQVSNAIFPQVEAAQKMSNRAVAVLRGETVTGTIWITQKS...


In [3]:
x_train_seq = train_df["Sequence"].tolist()
y_train_labels = train_df["Label"].tolist()

x_test_seq = test_df["Sequence"].tolist()
y_test_labels = test_df["Label"].tolist()

from sklearn.linear_model import LogisticRegression
## defining model
model = LogisticRegression()




In [9]:

import dgeb
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
## first input which is just the embeddings i am using dgeb embedding model to make each seq into a vector
embedding_model = dgeb.get_model("facebook/esm2_t6_8M_UR50D")



INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/esm2_t6_8M_UR50D/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/esm2_t6_8M_UR50D/c731040fcd8d73dceaa04b0a8e6329b345b0f5df/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 5236.98it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

In [8]:
## converting train data to numerical vectors
x_train_input1 = embedding_model.encode(x_train_seq)[:, -1, :]

NameError: name 'embedding_model' is not defined

In [13]:
## doing same for test data
## broke into two cells so we would avoid rerunning
x_test_input1 = embedding_model.encode(x_test_seq)[:, -1, :]

encoding: 100%|██████████| 1/1 [02:05<00:00, 125.34s/it]


In [4]:
from sklearn.metrics import accuracy_score, f1_score 
## checking shape
print(x_train_input1.shape)
print(x_test_input1.shape)
## now fitting model on this info
model.fit(x_train_input1, y_train_labels)
predictions_input1 = model.predict(x_test_input1)

## accuracy and f1 score for input 1
print("accuracy:", accuracy_score(y_test_labels, predictions_input1))
print("f1:", f1_score(y_test_labels, predictions_input1, average="macro"))

NameError: name 'x_train_input1' is not defined

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score 
## now doing input 2 which is k mer counts of 3

## defining a function to make kmer
def kmerFunction(seq):
    return " ".join([seq[i:i+3] for i in range(len(seq)-2)])


train_kmer = [kmerFunction(seq) for seq in x_train_seq]
test_kmer = [kmerFunction(seq) for seq in x_test_seq]

## now counting how many times each kmer shows up
helper = CountVectorizer()
x_train_input2 = helper.fit_transform(train_kmer)
x_test_input2 = helper.transform(test_kmer)

## checking shape
print(x_train_input2.shape)
print(x_test_input2.shape)

## defining model again

model2 = LogisticRegression()
model2.fit(x_train_input2, y_train_labels)
predictions_input2 = model2.predict(x_test_input2)

## printing accuracy and f1 score
print("accuracy:", accuracy_score(y_test_labels, predictions_input2))
print("f1:", f1_score(y_test_labels, predictions_input2, average="macro"))

(512, 7986)
(128, 7986)
accuracy: 0.2109375
f1: 0.1513206845238095


In [7]:
## now doing input 3 which is weighted kmer
from sklearn.feature_extraction.text import TfidfVectorizer

## defining the weighted helper
weighted =  TfidfVectorizer()

x_train_input3 = weighted.fit_transform(train_kmer)
x_test_input3 = weighted.transform(test_kmer)

## ensuring shape is accurate
print(x_train_input3.shape)
print(x_test_input3.shape)

## making model3
model3 = LogisticRegression()

model3.fit(x_train_input3, y_train_labels)
predictions_input3 = model3.predict(x_test_input3)

## f1 and accuracy scores
print("accuracy:", accuracy_score(y_test_labels, predictions_input3))
print("f1:", f1_score(y_test_labels, predictions_input3, average="macro"))

(512, 7986)
(128, 7986)
accuracy: 0.25
f1: 0.20703124999999997
